# Lab 2 — Music Genre Classification with CNNs

This notebook implements both tasks in the lab guide: (1) compare simple CNN configurations using the provided mel-spectrogram images, and (2) fine-tune a pretrained VGG16 model. Both tasks use the same shuffled, stratified 80/20 train/test split. The test set is held out until final evaluation.

## 1. Setup and reproducibility

The notebook is designed for Google Colab or Jupyter. A GPU is recommended, especially for VGG16.

In [ ]:
# If needed, uncomment the install line in a fresh environment.
# %pip install -q tensorflow scikit-learn matplotlib seaborn pillow pandas

import os, random, json, time, pathlib, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from pathlib import Path

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print('TensorFlow:', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))

## 2. Load the spectrogram dataset

Download the files linked in the lab handout. Extract the `images_original` directory so that it contains one subdirectory per genre (for example, `images_original/blues/`). Set `DATA_DIR` below to that directory. Do not use the CSV global descriptors for these CNN experiments; the task specifically asks for the provided spectrogram images.

In [ ]:
# Option A (Google Colab): upload a ZIP containing images_original/ and extract it.
# from google.colab import files
# uploaded = files.upload()  # choose your dataset zip
# zip_name = next(iter(uploaded))
# with zipfile.ZipFile(zip_name) as zf: zf.extractall('/content/music_genre_data')
# DATA_DIR = '/content/music_genre_data/images_original'

# Option B (local Jupyter): point this to the extracted images_original folder.
DATA_DIR = 'data/images_original'  # CHANGE THIS PATH
DATA_DIR = Path(DATA_DIR)
if not DATA_DIR.exists():
    raise FileNotFoundError(f'Could not find {DATA_DIR.resolve()}. Set DATA_DIR to the extracted images_original directory.')

VALID_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}
records = []
for genre_dir in sorted(p for p in DATA_DIR.iterdir() if p.is_dir()):
    for image_path in sorted(genre_dir.rglob('*')):
        if image_path.is_file() and image_path.suffix.lower() in VALID_EXTS:
            records.append({'path': str(image_path), 'genre': genre_dir.name})
df = pd.DataFrame(records)
if df.empty:
    raise ValueError('No images found. Expected image files inside one subfolder per genre.')
print('Images:', len(df), '| Genres:', df['genre'].nunique())
display(df.groupby('genre').size().rename('images').to_frame())
display(df.head())

In [ ]:
# Visual sanity check
fig, axes = plt.subplots(1, min(5, len(df)), figsize=(15, 3))
for ax, (_, row) in zip(np.atleast_1d(axes), df.sample(min(5, len(df)), random_state=SEED).iterrows()):
    ax.imshow(plt.imread(row['path']))
    ax.set_title(row['genre']); ax.axis('off')
plt.tight_layout(); plt.show()

## 3. Shared shuffled 80/20 train/test split

The split is stratified to preserve genre proportions. The 20% test set is identical for both model families. Model selection uses only a validation subset of the 80% training partition.

In [ ]:
train_df, test_df = train_test_split(
    df, test_size=0.20, random_state=SEED, shuffle=True, stratify=df['genre']
)
# Create validation data only from the training partition (validation = 20% of train).
train_df, val_df = train_test_split(
    train_df, test_size=0.20, random_state=SEED, shuffle=True, stratify=train_df['genre']
)
class_names = sorted(df['genre'].unique())
num_classes = len(class_names)
class_to_id = {name: i for i, name in enumerate(class_names)}
print(f'Train: {len(train_df)} ({len(train_df)/len(df):.1%} of all data)')
print(f'Validation: {len(val_df)} ({len(val_df)/len(df):.1%} of all data)')
print(f'Test: {len(test_df)} ({len(test_df)/len(df):.1%} of all data)')
print('Classes:', class_names)

## 4. Task 1 — train and compare simple CNNs

All configurations use the same training/validation split and input size. The small grid varies the number of convolutional blocks and filters. Early stopping selects the best epoch using validation loss; the test set is not used to choose the architecture.

In [ ]:
IMG_SIZE = (128, 128)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def make_dataset(frame, training=False, image_size=IMG_SIZE, batch_size=BATCH_SIZE):
    paths = frame['path'].astype(str).to_numpy()
    labels = frame['genre'].map(class_to_id).to_numpy(dtype=np.int32)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    def load_image(path, label):
        image = tf.io.read_file(path)
        image = tf.io.decode_image(image, channels=3, expand_animations=False)
        image.set_shape([None, None, 3])
        image = tf.image.resize(image, image_size)
        image = tf.cast(image, tf.float32) / 255.0
        return image, label
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    if training: ds = ds.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(batch_size).prefetch(AUTOTUNE)

train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df)

def build_simple_cnn(filters=(32, 64), dropout=0.35):
    inputs = tf.keras.Input(shape=(*IMG_SIZE, 3))
    x = inputs
    for f in filters:
        x = tf.keras.layers.Conv2D(f, 3, padding='same', activation='relu')(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.MaxPooling2D()(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(dropout)(x)
    x = tf.keras.layers.Dense(128, activation='relu')(x)
    x = tf.keras.layers.Dropout(dropout / 2)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
    model = tf.keras.Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

cnn_configs = [
    {'name': 'CNN_2blocks_32_64', 'filters': (32, 64)},
    {'name': 'CNN_2blocks_32_128', 'filters': (32, 128)},
    {'name': 'CNN_3blocks_32_64_128', 'filters': (32, 64, 128)},
]
cnn_results = []
cnn_models = {}
for config in cnn_configs:
    print('\nTraining', config['name'])
    model = build_simple_cnn(config['filters'])
    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-5)
    ]
    start = time.time()
    history = model.fit(train_ds, validation_data=val_ds, epochs=30, callbacks=callbacks, verbose=1)
    elapsed = time.time() - start
    val_loss, val_acc = model.evaluate(val_ds, verbose=0)
    cnn_models[config['name']] = model
    cnn_results.append({'model': config['name'], 'filters': str(config['filters']),
                        'best_val_accuracy': float(val_acc), 'best_val_loss': float(val_loss),
                        'training_seconds': round(elapsed, 1), 'epochs_run': len(history.history['loss'])})
cnn_results_df = pd.DataFrame(cnn_results).sort_values('best_val_accuracy', ascending=False)
display(cnn_results_df)

In [ ]:
# Select architecture using validation accuracy only, then evaluate on held-out test data.
best_cnn_name = cnn_results_df.iloc[0]['model']
best_cnn = cnn_models[best_cnn_name]
cnn_test_loss, cnn_test_acc = best_cnn.evaluate(test_ds, verbose=0)
print('Selected CNN:', best_cnn_name)
print(f'Test accuracy: {cnn_test_acc:.4f} | Test loss: {cnn_test_loss:.4f}')

def evaluate_classifier(model, dataset, labels_for_rows, model_name):
    y_true = np.concatenate([y.numpy() for _, y in dataset], axis=0)
    probs = model.predict(dataset, verbose=0)
    y_pred = np.argmax(probs, axis=1)
    print(f'\n{model_name} — test classification report')
    print(classification_report(y_true, y_pred, labels=np.arange(num_classes), target_names=class_names, zero_division=0))
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(num_classes))
    plt.figure(figsize=(9, 7)); sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names, cmap='Blues')
    plt.xlabel('Predicted genre'); plt.ylabel('True genre'); plt.title(f'{model_name} — test confusion matrix'); plt.tight_layout(); plt.show()
    return {'model': model_name, 'test_accuracy': accuracy_score(y_true, y_pred), 'test_loss': float(model.evaluate(dataset, verbose=0)[0])}

final_results = [evaluate_classifier(best_cnn, test_ds, test_df, best_cnn_name)]

## 5. Task 2 — fine-tune a pretrained model (VGG16)

The lab gives VGGish as an example of a pretrained model. Here we use ImageNet-pretrained VGG16, a widely available deep CNN, as a transfer-learning baseline for the provided RGB spectrogram images. The convolutional backbone is initially frozen while a new genre classifier is trained; then the final convolutional block is unfrozen and fine-tuned at a low learning rate. This is a spectrogram-image transfer-learning experiment, not the audio-waveform VGGish model.

In [ ]:
# VGG16 expects the usual 224x224 image input and ImageNet preprocessing.
VGG_SIZE = (224, 224)
VGG_BATCH_SIZE = 16
def make_vgg_dataset(frame, training=False):
    paths = frame['path'].astype(str).to_numpy()
    labels = frame['genre'].map(class_to_id).to_numpy(dtype=np.int32)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    def load_image(path, label):
        image = tf.io.read_file(path)
        image = tf.io.decode_image(image, channels=3, expand_animations=False)
        image.set_shape([None, None, 3])
        image = tf.image.resize(image, VGG_SIZE)
        image = tf.cast(image, tf.float32) * 255.0
        image = tf.keras.applications.vgg16.preprocess_input(image)
        return image, label
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    if training: ds = ds.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(VGG_BATCH_SIZE).prefetch(AUTOTUNE)

vgg_train_ds = make_vgg_dataset(train_df, training=True)
vgg_val_ds = make_vgg_dataset(val_df)
vgg_test_ds = make_vgg_dataset(test_df)

base = tf.keras.applications.VGG16(include_top=False, weights='imagenet', input_shape=(*VGG_SIZE, 3))
base.trainable = False
inputs = tf.keras.Input(shape=(*VGG_SIZE, 3))
x = base(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(256, activation='relu')(x)
x = tf.keras.layers.Dropout(0.4)(x)
outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
vgg_model = tf.keras.Model(inputs, outputs)
vgg_model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
vgg_callbacks = [tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)]
vgg_history_head = vgg_model.fit(vgg_train_ds, validation_data=vgg_val_ds, epochs=8, callbacks=vgg_callbacks)

# Fine-tune only the last VGG16 convolutional block; keep BatchNorm-free backbone stable.
base.trainable = True
for layer in base.layers:
    layer.trainable = layer.name.startswith('block5_')
vgg_model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
vgg_history_finetune = vgg_model.fit(vgg_train_ds, validation_data=vgg_val_ds, epochs=8, callbacks=vgg_callbacks)
vgg_test_loss, vgg_test_acc = vgg_model.evaluate(vgg_test_ds, verbose=0)
print(f'VGG16 test accuracy: {vgg_test_acc:.4f} | Test loss: {vgg_test_loss:.4f}')
final_results.append(evaluate_classifier(vgg_model, vgg_test_ds, test_df, 'VGG16_finetuned'))

In [ ]:
# Compare the selected CNN and fine-tuned pretrained model on the same held-out test partition.
results_df = pd.DataFrame(final_results).sort_values('test_accuracy', ascending=False)
display(results_df)
results_df.to_csv('lab2_test_results.csv', index=False)
cnn_results_df.to_csv('lab2_cnn_configuration_results.csv', index=False)
print('Saved lab2_test_results.csv and lab2_cnn_configuration_results.csv')

## 6. Discussion and Lab 1 comparison

Use the measured results above to complete the discussion; do not assume a result in advance.

- Report the validation scores for every CNN configuration and explain the selection criterion.
- Report test accuracy, loss, classification report and confusion matrix for the selected CNN and VGG16.
- Discuss which genres are most frequently confused, using the confusion matrices as evidence.
- Compare these results with the manually extracted-feature models from Lab 1. Include the actual Lab 1 scores, the split/evaluation protocol, computational cost and any differences in overfitting. The Lab 1 CSV features and the spectrogram CNNs represent different input representations, so state this when interpreting the comparison.
- Because the test set is held out from architecture selection, use test performance only for final reporting, not further tuning.

## 7. Reproducibility notes

- Keep `SEED = 42` unchanged to reproduce the same split.
- The first run of VGG16 downloads pretrained ImageNet weights; internet access is required for that download.
- Results can vary slightly with TensorFlow version, hardware, and nondeterministic GPU operations.
- The lab dataset contains original spectrogram images; verify folder names and image counts before training.
- The notebook does not modify or depend on files inside `Lab1/`.